# Homework 01: Predicting Directional Reaching Performance

**Computational Sensorimotor Control** | Week 1 | Module 1: The Biological Plant

**Due:** One week from assignment | **Total:** 100 points

---

## The Experiment

A participant sits with their right arm in the horizontal plane. Their hand starts at a central position. On each trial, a target appears at one of **8 equally spaced directions** (0°, 45°, 90°, … , 315°) on a circle of radius 10 cm centered on the start position.

Your task: use the `Arm` class from Lab 01 to **design this experiment computationally** and **generate predictions** about which reach directions should be kinematically easiest vs. hardest. You will analyze two specific target directions in depth.

This is how computational models are used in practice — you build the model first, generate predictions, and then compare with experimental data.

---
## Part 0: Setup (0 pts)

> 📖 **Lecture notes:** §1.1 *Defining the Model* — the link lengths ($l_1=0.34$, $l_2=0.46$ m,
> Gribble et al. 1998) and anatomical limits ($\theta_1\in[0°,140°]$, $\theta_2\in[0°,145°]$)
> hard-coded below. The `Arm` class implements §2 (FK), §3 (IK) and §4 (the Jacobian).

In [ ]:
# === Setup ===
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (8, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

THETA1_LIM = (np.radians(0), np.radians(140))
THETA2_LIM = (np.radians(0), np.radians(145))

print('Setup complete.')

### Arm Class

The completed `Arm` class from Lab 01 is **provided below** — `forward_kinematics`,
`elbow_position`, `inverse_kinematics`, `jacobian`, `is_valid` — so that this assignment tests
experiment design and prediction rather than re-testing Lab 01. Just run the cell.

If you would rather use your own Lab 01 implementation, you are welcome to replace it, but check
that it passes the Lab 01 tests first: a bug in `inverse_kinematics` will silently propagate
through Parts 1–4.

In [ ]:
# Reference Arm class from Lab 01 — provided complete so you can focus on the
# analysis below. (If you prefer, replace it with your own Lab 01 implementation.)
class Arm:
    """2-link planar arm in the horizontal plane (Gribble et al. 1998 parameters)."""

    def __init__(self, l1=0.34, l2=0.46):
        self.l1, self.l2 = l1, l2
        self.theta1_lim = (np.radians(0), np.radians(140))
        self.theta2_lim = (np.radians(0), np.radians(145))

    def forward_kinematics(self, q):
        theta1, theta2 = q
        x = self.l1 * np.cos(theta1) + self.l2 * np.cos(theta1 + theta2)
        y = self.l1 * np.sin(theta1) + self.l2 * np.sin(theta1 + theta2)
        return np.array([x, y])

    def elbow_position(self, q):
        theta1, theta2 = q
        return np.array([self.l1 * np.cos(theta1), self.l1 * np.sin(theta1)])

    def inverse_kinematics(self, p):
        x, y = p
        r_sq = x**2 + y**2
        c2 = (r_sq - self.l1**2 - self.l2**2) / (2 * self.l1 * self.l2)
        if abs(c2) > 1.0:
            return []
        solutions = []
        for sign in [1, -1]:
            theta2 = sign * np.arccos(c2)
            k1 = self.l1 + self.l2 * np.cos(theta2)
            k2 = self.l2 * np.sin(theta2)
            theta1 = np.arctan2(y, x) - np.arctan2(k2, k1)
            q = np.array([theta1, theta2])
            if self.is_valid(q):
                solutions.append(q)
        return solutions

    def jacobian(self, q):
        theta1, theta2 = q
        s1, c1 = np.sin(theta1), np.cos(theta1)
        s12, c12 = np.sin(theta1 + theta2), np.cos(theta1 + theta2)
        return np.array([
            [-self.l1 * s1 - self.l2 * s12, -self.l2 * s12],
            [ self.l1 * c1 + self.l2 * c12,  self.l2 * c12]
        ])

    def is_valid(self, q):
        t1, t2 = q
        # ±1e-9 tolerance so exact-limit postures (e.g. θ₂ = 145°) are not rejected by rounding
        return (self.theta1_lim[0] - 1e-9 <= t1 <= self.theta1_lim[1] + 1e-9 and
                self.theta2_lim[0] - 1e-9 <= t2 <= self.theta2_lim[1] + 1e-9)

### Provided Helpers — No Action Needed

The functions below handle plotting and two small utilities, so that each task box asks you to write **at most two lines** of analysis code. Read them if you're curious, but you do not need to modify them.

- `draw_arm(arm, q, ax, ...)` — draw the arm at configuration `q`
- `draw_manip_ellipsoid(ax, arm, q, ...)` — draw the velocity manipulability ellipsoid at `q`
- `closest_target(axis_angle, directions)` — index of the target direction nearest an ellipsoid axis
- `condition_number(arm, q)` — Jacobian condition number √(λ_max/λ_min) at `q`

In [ ]:
# === Provided helpers (no action needed) ===

def draw_arm(arm, q, ax, color='#2E86AB', lw=3, alpha=1.0):
    """Draw the 2-link arm at configuration q onto axis ax."""
    elb = arm.elbow_position(q)
    hand = arm.forward_kinematics(q)
    ax.plot([0, elb[0]], [0, elb[1]], '-', color='#1B2A4A', lw=lw, solid_capstyle='round', alpha=alpha)
    ax.plot([elb[0], hand[0]], [elb[1], hand[1]], '-', color=color, lw=lw, solid_capstyle='round', alpha=alpha)
    ax.plot(0, 0, 'ko', ms=10, zorder=10)
    ax.plot(*elb, 'o', color='gray', ms=8, zorder=10, alpha=alpha)
    ax.plot(*hand, 'o', color=color, ms=8, zorder=10, alpha=alpha)

def draw_manip_ellipsoid(ax, arm, q, scale=1.0, color='#E74C3C', alpha=0.15, show_axes=False):
    """Draw the velocity manipulability ellipsoid (A = J Jᵀ) at configuration q."""
    J = arm.jacobian(q)
    evals, evecs = np.linalg.eigh(J @ J.T)
    order = np.argsort(evals)[::-1]
    evals, evecs = evals[order], evecs[:, order]
    center = arm.forward_kinematics(q)
    width  = 2 * np.sqrt(max(evals[0], 0)) * scale
    height = 2 * np.sqrt(max(evals[1], 0)) * scale
    angle  = np.degrees(np.arctan2(evecs[1, 0], evecs[0, 0]))
    ax.add_patch(Ellipse(center, width, height, angle=angle, fc=color, alpha=alpha, ec=color, lw=2))
    if show_axes:
        for i, (c, lab) in enumerate(zip(['#27AE60', '#E74C3C'], ['major (easy)', 'minor (hard)'])):
            vec = evecs[:, i] * np.sqrt(evals[i]) * scale
            for s in (+1, -1):
                ax.annotate('', xy=(center[0] + s * vec[0], center[1] + s * vec[1]), xytext=center,
                            arrowprops=dict(arrowstyle='->', color=c, lw=2.5))
            ax.text(center[0] + vec[0] * 1.15, center[1] + vec[1] * 1.15, lab,
                    color=c, fontsize=10, fontweight='bold', ha='center')

def closest_target(axis_angle, directions):
    """Index of the target direction nearest an ellipsoid axis (axes are 180°-symmetric)."""
    diffs = [abs(((d - axis_angle) + 180) % 360 - 180) for d in directions]
    return int(np.argmin(diffs))

def condition_number(arm, q):
    """Jacobian condition number sqrt(lambda_max / lambda_min) of A = J Jᵀ at q."""
    evals = np.linalg.eigvalsh(arm.jacobian(q) @ arm.jacobian(q).T)
    return np.sqrt(max(evals) / min(evals))

print('Helpers loaded: draw_arm, draw_manip_ellipsoid, closest_target, condition_number')

---
## Part 1: Experiment Design (20 pts)

> 📖 **Lecture notes:** §2.1 *Geometric Derivation* — Eqs. 2.2a–b are the forward kinematics you
> call in Task 1.1 · §2.3 *The Workspace* and **Notes Figure 3** — the crescent that decides which
> targets are reachable in Task 1.2 · §3.1 *Solving for θ₂* — why a target is unreachable when
> $r > l_1+l_2$, and why only $\theta_2 > 0$ counts as a valid solution.


### Task 1.1 — Choose a Start Posture (5 pts)

A common start posture in reaching experiments places the hand roughly at the body midline, with the elbow at a comfortable angle. Use $\theta_1 = 55°$, $\theta_2 = 75°$.

**You write 1 line:** compute the start hand position with forward kinematics. (The posture, the print statements, and the validity check are provided.)

In [ ]:
arm = Arm()
q_start = np.array([np.radians(55), np.radians(75)])

# TODO (1 line): compute the start hand position with forward kinematics
start_pos = ...  # your code here
print(f"Start posture: \u03b8\u2081 = {np.degrees(q_start[0]):.1f}\u00b0, \u03b8\u2082 = {np.degrees(q_start[1]):.1f}\u00b0")
print(f"Hand position: ({start_pos[0]:.4f}, {start_pos[1]:.4f}) m")
print(f"Valid: {arm.is_valid(q_start)}")

### Task 1.2 — Place the 8 Targets (10 pts)

Place 8 targets at 10 cm (0.10 m) from the start position, equally spaced at 0°, 45°, …, 315° (0° = rightward, 90° = forward/upward in our frame).

**You write 2 lines:**
1. the position of each target (start position + a 10 cm step in the given direction), and
2. whether each target is reachable (has at least one valid IK solution).

The plot (arm + targets colored by reachability) and the reachability printout are provided.

In [ ]:
reach_radius = 0.10                      # 10 cm
directions = np.arange(0, 360, 45)       # 0, 45, ..., 315 degrees
dir_rad = np.radians(directions)

# TODO (1 line): target i sits 10 cm from start_pos in direction `angle`
targets = np.zeros((8, 2))
for i, angle in enumerate(dir_rad):
    targets[i] = ...  # your code here

# TODO (1 line): a target is reachable if inverse_kinematics returns at least one solution
reachable = []
for i, t in enumerate(targets):
    sols = arm.inverse_kinematics(t)
    reachable.append(...)  # your code here (hint: len(sols) > 0)

# --- Plotting provided (uses the draw_arm helper) ---
fig, ax = plt.subplots(figsize=(9, 9))
draw_arm(arm, q_start, ax)
for i, (t, r) in enumerate(zip(targets, reachable)):
    color = '#27AE60' if r else '#E74C3C'
    ax.plot(*t, 'o', color=color, ms=12, zorder=5)
    ax.annotate(f'{directions[i]}\u00b0', xy=t, xytext=(t[0]+0.015, t[1]+0.015), fontsize=10, fontweight='bold', color=color)
ax.plot(*start_pos, 's', color='#E8553A', ms=10, zorder=10, label='Start')
ax.add_patch(plt.Circle(start_pos, reach_radius, fill=False, ls='--', color='gray', alpha=0.4))
ax.set_aspect('equal'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Center-Out Reaching: 8 Target Directions', fontweight='bold'); ax.legend()
plt.show()

for i, d in enumerate(directions):
    status = "\u2713 reachable" if reachable[i] else "\u2717 unreachable"
    print(f"  {d:3d}\u00b0: ({targets[i,0]:.4f}, {targets[i,1]:.4f})  {status}")

### Question 1.1 (5 pts)

Are all 8 targets reachable? If any are not, explain why in terms of the workspace geometry from the lecture notes.

*Your answer here:*


---
## Part 2: Manipulability Analysis at the Start Posture (25 pts)

> 📖 **Lecture notes:** §5.1 *Construction* — Eqs. 5.1–5.4 derive the ellipsoid from
> $\|\dot\theta\|=1$ · §5.2 *Eigenvalue Analysis* — eigenvectors give the principal axes you
> identify in Task 2.2; eigenvalues give squared semi-axis lengths ($\lambda_i=\sigma_i^2$) ·
> §5.3 *Yoshikawa’s Manipulability Measure* — Eq. 5.5, $w=|l_1 l_2 \sin\theta_2|$ ·
> **Notes Figures 6–8** (ellipsoid shape vs. posture, and across the workspace).


> **Convention note — $JJ^T$ does not mean the same thing here as in the lecture notes.**
> The notes write each ellipsoid as a **quadratic form**; this assignment (like Lab 01) builds its
> **shape matrix** $A$, whose eigenvalue square roots are the semi-axis lengths. Same ellipsoids,
> different algebra:
>
> | ellipsoid | quadratic form (notes) | shape matrix $A$ (here) | semi-axes |
> |---|---|---|---|
> | velocity | $\dot p^{T}(JJ^{T})^{-1}\dot p \le 1$ | `J @ J.T` | $\sigma_i$ |
> | force | $F^{T}(JJ^{T})\,F \le 1$ | `inv(J).T @ inv(J)` $=(JJ^{T})^{-1}$ | $1/\sigma_i$ |
>
> So $A = JJ^T$ below is the **velocity** (manipulability) ellipsoid, even though $JJ^{T}$ names
> the **force** ellipsoid's quadratic form in the notes. Everything in this assignment is the
> velocity ellipsoid.

### Task 2.1 — Compute and Plot the Manipulability Ellipsoid (10 pts)

**You write 2 lines:** build the manipulability matrix $A = JJ^T$ at the start posture, then eigendecompose it. (The Jacobian call, the eigenvalue sorting, the derived condition number and Yoshikawa's $w$, the printout, and the ellipsoid plot are all provided.)

In [ ]:
# TODO (2 lines): build the manipulability matrix A = J J^T, then eigendecompose it
J = arm.jacobian(q_start)
A = ...  # your code here (line 1): A = J J^T
eigenvalues, eigenvectors = ...  # your code here (line 2): use np.linalg.eigh(A)

# --- Provided: order eigenpairs (largest first) and derive the reported quantities ---
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]
w = abs(np.linalg.det(J))                          # Yoshikawa's manipulability
cond = np.sqrt(eigenvalues[0] / eigenvalues[1])    # condition number (anisotropy)

print(f"Eigenvalues: \u03bb\u2081 = {eigenvalues[0]:.6f}, \u03bb\u2082 = {eigenvalues[1]:.6f}")
print(f"Condition number: {cond:.2f}")
print(f"Yoshikawa's w: {w:.6f}")

# --- Plotting provided (draw_arm + draw_manip_ellipsoid helpers) ---
fig, ax = plt.subplots(figsize=(9, 9))
draw_arm(arm, q_start, ax)
draw_manip_ellipsoid(ax, arm, q_start, scale=1.0, show_axes=True)
for i, (t, r) in enumerate(zip(targets, reachable)):
    ax.plot(*t, 'o', color='#27AE60' if r else '#E74C3C', ms=10)
    ax.annotate(f'{directions[i]}\u00b0', xy=t, xytext=(t[0]+0.012, t[1]+0.012), fontsize=9)
ax.set_aspect('equal'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Manipulability Ellipsoid at Start Posture', fontweight='bold')
plt.show()

### Task 2.2 — Identify the Easy and Hard Directions (10 pts)

The major axis of the ellipsoid is the direction the hand can move most easily (largest velocity for a given joint speed); the minor axis is the hardest.

**You write 2 lines:** using the provided `closest_target(axis_angle, directions)` helper, find which of the 8 targets is closest to the easy (major) axis and which is closest to the hard (minor) axis. (The axis angles and the printout are provided.)

In [ ]:
# Provided: the principal-axis directions of the ellipsoid
major_axis = eigenvectors[:, 0]
minor_axis = eigenvectors[:, 1]
easy_angle = np.degrees(np.arctan2(major_axis[1], major_axis[0]))
hard_angle = np.degrees(np.arctan2(minor_axis[1], minor_axis[0]))
print(f"Easy direction (major axis): {easy_angle:.1f}\u00b0")
print(f"Hard direction (minor axis): {hard_angle:.1f}\u00b0")

# TODO (2 lines): use the provided closest_target(axis_angle, directions) helper to find
# which of the 8 target directions is nearest the easy axis and which is nearest the hard axis
easy_target_idx = ...  # your code here (line 1)
hard_target_idx = ...  # your code here (line 2)

print(f"\nClosest target to easy axis: {directions[easy_target_idx]}\u00b0 (target {easy_target_idx})")
print(f"Closest target to hard axis: {directions[hard_target_idx]}\u00b0 (target {hard_target_idx})")

### Question 2.1 (5 pts)

In your own words, explain what the manipulability ellipsoid tells us about reaching in the easy vs. hard direction. What specific kinematic quantity is larger/smaller in each direction?

*Your answer here:*


---
## Part 3: Detailed Predictions for Two Reach Directions (35 pts)

> 📖 **Lecture notes:** §3 *Inverse Kinematics* — Eqs. 3.1–3.8, solved at each target in
> Task 3.1 · §4.3 *The Jacobian as a Velocity Transformer* — Eqs. 4.6–4.7 explain why the same
> 10 cm hand displacement costs different joint excursions · §4.2 *The Determinant and
> Singularities* — Eqs. 4.4–4.5; the condition number rises as $\theta_2 \to 0$ ·
> §5.2 *Eigenvalue Analysis* for reading the ellipsoid change between start and target.


Now analyze the two selected reaches in detail: one toward the **easy target** and one toward the **hard target**.

### Task 3.1 — Compute Joint-Level Requirements (15 pts)

Inside the provided loop (which solves IK for each target), **you write 2 lines:** the joint displacement $\Delta q = q_{target} - q_{start}$ and its magnitude $\|\Delta q\|$. (The IK call, the target-posture condition number via the `condition_number` helper, and the comparison table are provided.)

In [ ]:
results = {}

for label, idx in [('Easy', easy_target_idx), ('Hard', hard_target_idx)]:
    target = targets[idx]
    sols = arm.inverse_kinematics(target)           # provided
    q_target = sols[0]                              # first valid solution

    # TODO (2 lines): joint displacement from start to target, and its magnitude ||Delta q||
    delta_q = ...  # your code here (line 1)
    excursion = ...  # your code here (line 2): np.linalg.norm(...)

    cond_target = condition_number(arm, q_target)   # provided helper
    results[label] = {'direction': directions[idx], 'target': target, 'q_target': q_target,
                      'delta_q': delta_q, 'excursion': excursion, 'cond_target': cond_target}

# --- Comparison table (provided) ---
print(f"{'':15s} {'Easy':>12s} {'Hard':>12s}")
print(f"{'-'*40}")
print(f"{'Direction':15s} {results['Easy']['direction']:>10d}\u00b0  {results['Hard']['direction']:>10d}\u00b0")
print(f"{'\u0394\u03b8\u2081':15s} {np.degrees(results['Easy']['delta_q'][0]):>10.1f}\u00b0  {np.degrees(results['Hard']['delta_q'][0]):>10.1f}\u00b0")
print(f"{'\u0394\u03b8\u2082':15s} {np.degrees(results['Easy']['delta_q'][1]):>10.1f}\u00b0  {np.degrees(results['Hard']['delta_q'][1]):>10.1f}\u00b0")
print(f"{'\u2016\u0394q\u2016 (rad)':15s} {results['Easy']['excursion']:>11.3f}  {results['Hard']['excursion']:>11.3f}")
print(f"{'Cond # (start)':15s} {cond:>11.2f}  {cond:>11.2f}")
print(f"{'Cond # (target)':15s} {results['Easy']['cond_target']:>11.2f}  {results['Hard']['cond_target']:>11.2f}")

### Task 3.2 — Visualize Both Reaches (10 pts)

The cell below builds a two-panel figure (easy reach | hard reach), drawing the start posture, the hand path, and the titles for you.

**You write 2 lines:** inside the loop, draw the arm at the **target** posture and its manipulability ellipsoid (use `data['q_target']` with the `draw_arm` and `draw_manip_ellipsoid` helpers).

In [ ]:
# Two-panel comparison of the easy and hard reaches (Figure 1).
# The scaffold draws the start posture, hand path, and titles for you.
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, (label, data) in zip(axes, results.items()):
    # Provided: start posture (light) and its ellipsoid
    draw_arm(arm, q_start, ax, color='#87CEEB', lw=2, alpha=0.4)
    draw_manip_ellipsoid(ax, arm, q_start, scale=0.8, color='#999999', alpha=0.10)

    # TODO (2 lines): draw the TARGET posture (bold) and its manipulability ellipsoid, using data['q_target'].
    #   Suggested styling: draw_arm(..., color='#2E86AB', lw=4);  draw_manip_ellipsoid(..., scale=0.8, color='#E74C3C', alpha=0.20)
    # your code here (line 1)
    # your code here (line 2)

    # Provided: straight hand path, target marker, axes, title
    ax.plot([start_pos[0], data['target'][0]], [start_pos[1], data['target'][1]], '--', color='#E8553A', lw=2, alpha=0.6)
    ax.plot(*data['target'], 's', color='#E8553A', ms=12, zorder=10)
    ax.set_aspect('equal'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_title(f"{label} direction: {data['direction']}\u00b0\n\u2016\u0394q\u2016 = {data['excursion']:.3f} rad", fontweight='bold', fontsize=13)
    ax.set_xlim(-0.1, 0.6); ax.set_ylim(-0.1, 0.7)

plt.suptitle('Figure 1: Easy vs. Hard Reach Directions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Question 3.1 (10 pts)

Compare your two reaches across the following dimensions. For each, state which direction (easy or hard) has the larger value, and explain *why* in terms of the Jacobian and the ellipsoid geometry.

**(a)** Total joint excursion $\|\Delta q\|$

**(b)** Change in the Jacobian condition number from start to target

**(c)** If we assume movement duration is proportional to joint excursion (a simplification), which reach would be faster? Is this consistent with experimental findings that reaching performance varies with direction?

*Your answer here:*


---
## Part 4: Methods & Predictions Write-Up (20 pts)

> 📖 **Lecture notes:** §7 *Summary of Key Results* — every equation you will cite, in one place ·
> §6.3 *Preview: Why This Matters* — useful framing for the Predictions paragraph ·
> **Required reading:** Gribble et al. (1998) for the arm model, Morasso (1981) for the reaching
> literature your predictions speak to.


Write a short **Methods and Predictions** section (300–500 words) as if for a real paper. This is the core skill: translating a computational analysis into a scientific narrative.

Your write-up should include:

1. **Methods** (~150 words): Describe the simulated experiment — the arm model (cite Gribble et al. 1998), the start posture, the target layout, how you selected the easy and hard directions, and what kinematic quantities you computed.

2. **Predictions** (~150–300 words): State your predictions clearly. Which direction should produce faster/more accurate reaches? What is the predicted ratio of joint excursions? How does the manipulability change between start and target for each direction? Reference your figures and table by number.

3. **One figure**: Include your two-panel comparison from Task 3.2 (or an improved version) as "Figure 1" and reference it in the text.

*Format:* Write in third person, past tense ("The model predicted that…"), as you would in a journal submission.

*Your write-up here:*


---
## Summary

> 📖 **Lecture notes:** §7 *Summary of Key Results* and the **Notation Reference** at the end of
> §7 consolidate the conventions used throughout this assignment.


You designed a center-out reaching experiment computationally, used the manipulability ellipsoid to identify easy and hard reach directions, generated quantitative predictions about joint-level requirements, and communicated those predictions in a scientific format.

**Key takeaway:** The same 10 cm reach in task space can require very different joint excursions depending on direction — and the Jacobian tells you exactly why.

**A caveat worth carrying forward.** Reaching *is* direction-dependent experimentally — Gordon et al. (1994) found systematic extent errors that vary with reach direction. But they attributed those errors to the arm's **inertial** anisotropy (a property of the mass matrix $M(q)$), not to the **kinematic** anisotropy you computed here. The two are correlated — both come from limb geometry and posture — but they are different claims, and the data do not adjudicate between them. Keep your predictions kinematic, and say so explicitly.

**Next week:** We add muscles to the skeleton. The kinematic anisotropy you found here will be compounded by muscle-dependent force anisotropy — and once the arm has dynamics, you will be able to separate the kinematic and inertial accounts that this week's model cannot.